# Эмбеддинги CLIP для всех кадров BDD100K (Colab, T4)

Что нужно в `MyDrive/bdd100k/`:
- `bdd100k_images_100k.zip`;
- папка `check_cpu/` - эталон, посчитанный локально на CPU (`python -m src.embed --mode crop --limit 256 --device cpu --out artifacts/check_cpu`
  и то же с `--mode pad`): шесть файлов `clip_b32_{crop,pad}.npy`, `_names.parquet`, `_meta.json`.

Сначала те же 256 кадров считаются на GPU и сверяются с эталоном: если минимальный косинус не выше 0,999, ноутбук
останавливается. Потом полный проход по 100K кадров в двух вариантах (crop и pad). Он пишет на Drive куски
по 10 000 кадров: если сессия оборвётся, перезапустите все ячейки по порядку, готовые куски пропустятся.
Результат - `MyDrive/bdd100k/emb/`, локально его нужно положить в `artifacts/emb/` (папки `*_chunks` не нужны).

In [ ]:
import os
import subprocess

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/bdd100k"
REPO_URL = "https://github.com/artyombelyaev/road-frame-search.git"
REPO = "/content/road-frame-search"
ZIP = "/content/bdd100k_images_100k.zip"

In [ ]:
if not os.path.exists(REPO):
    !git clone -q $REPO_URL $REPO
else:
    !git -C $REPO pull -q
%cd $REPO
!git log --oneline -1
!pip install -q transformers==5.21.0
!python -c "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))"

In [ ]:
# читать 100K файлов с примонтированного Drive очень долго, поэтому архив копирую на диск Colab.
# Копия считается готовой, только если размер совпал с файлом на Drive и в архиве 100 000 кадров:
# оборванное копирование или ещё не загруженный на Drive файл иначе дают BadZipFile дальше.
import zipfile

SRC = f"{DRIVE_DIR}/bdd100k_images_100k.zip"


def zip_ok(path):
    try:
        with zipfile.ZipFile(path) as z:
            return sum(n.endswith(".jpg") for n in z.namelist()) == 100_000
    except zipfile.BadZipFile:
        return False


print("на Drive:", os.path.getsize(SRC), "байт")
if not (os.path.exists(ZIP) and os.path.getsize(ZIP) == os.path.getsize(SRC) and zip_ok(ZIP)):
    !rm -f $ZIP
    !cp "$SRC" $ZIP.part && mv $ZIP.part $ZIP
assert os.path.getsize(ZIP) == os.path.getsize(SRC), (os.path.getsize(ZIP), os.path.getsize(SRC))
assert zip_ok(ZIP), "архив битый: проверьте, что файл на Drive загрузился целиком"
print("копия:", os.path.getsize(ZIP), "байт, архив открывается, 100 000 кадров")

## 1. Сверка с CPU на 256 кадрах

In [ ]:
# вывод скрипта печатаю построчно: без этого Colab не показывает ни прогресс, ни текст ошибки
def run(*args):
    cmd = ["python", "-m", "src.embed", "--images-zip", ZIP, *args]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="")
    if proc.wait():
        raise RuntimeError(f"src.embed завершился с кодом {proc.returncode}")


for mode in ["crop", "pad"]:
    run("--mode", mode, "--limit", "256", "--out", "/content/check", "--compare-with", f"{DRIVE_DIR}/check_cpu")

## 2. Полный проход

In [ ]:
run("--mode", "crop", "--out", f"{DRIVE_DIR}/emb", "--workers", "2")

In [ ]:
run("--mode", "pad", "--out", f"{DRIVE_DIR}/emb", "--workers", "2")

In [ ]:
!ls -lh $DRIVE_DIR/emb
!cat $DRIVE_DIR/emb/clip_b32_crop_meta.json